In [ ]:
EXPERIMENT_ID = '2026-07-09_18-42-14_full'
WEIGHTS_ID = -1

TRAIN_PCA_SAMPLES = 500_000
TEST_PCA_SAMPLES = 100_000

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm
import os

from constants import DATA_ROOT, RESULTS_ROOT

In [ ]:
weights_list = os.listdir(RESULTS_ROOT / 'training' / EXPERIMENT_ID / 'weights')
weights_list = [w_name for w_name in weights_list if w_name.endswith('.weights.h5')]
weights_list.sort()
weights_to_load = weights_list[WEIGHTS_ID]
output_prefix = weights_to_load.split('.')[0]
print(output_prefix)

In [ ]:
SRC_DIR = DATA_ROOT / 'shuffled_shards_emb'
emb_shard_fnames = list(SRC_DIR.glob("*.npy"))
emb_shard_fnames.sort()

In [ ]:
train_embs = []
counter = 0

for emb_shard_f in emb_shard_fnames:
    this_emb = np.load(emb_shard_f)
    train_embs.append(this_emb)
    counter += this_emb.shape[0]
    if counter > TRAIN_PCA_SAMPLES:
        break

train_embs = np.concatenate(train_embs, axis=0)[:TRAIN_PCA_SAMPLES].astype('float64')

In [ ]:
emb_shard_fnames = emb_shard_fnames[::-1]

In [ ]:
test_embs = []
counter = 0

for emb_shard_f in emb_shard_fnames:
    this_emb = np.load(emb_shard_f)
    test_embs.append(this_emb)
    counter += this_emb.shape[0]
    if counter > TEST_PCA_SAMPLES:
        break

test_embs = np.concatenate(test_embs, axis=0)[:TEST_PCA_SAMPLES].astype('float32')

In [ ]:
from sklearn.decomposition import PCA

n_components = train_embs.shape[1]
pca = PCA(n_components=n_components, svd_solver="full", random_state=42)
pca.fit(train_embs)

In [ ]:
Z_test = pca.transform(test_embs)
total_var = np.var(test_embs, axis=0).sum()

test_cumvar = []
ks = np.linspace(1, n_components, 10).astype('int64')
for k in tqdm(ks):
    Z_trunc = Z_test.copy()
    Z_trunc[:, k:] = 0
    recon = pca.inverse_transform(Z_trunc)
    test_cumvar.append(1 - np.var(test_embs - recon, axis=0).sum() / total_var)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(ks, np.cumsum(pca.explained_variance_ratio_)[ks-1], label="Train")
plt.plot(ks, test_cumvar,linestyle='--', label="Test")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA – Train vs Test")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.ylim(0, 1)
plt.yticks(np.linspace(0, 1, 11))
plt.show()

In [ ]:
assert test_cumvar[-1] > 0.9999
test_cumvar[-1]

In [ ]:
np.save(RESULTS_ROOT / 'training' / EXPERIMENT_ID / 'weights' / f'{output_prefix}_pca_mat.npy', pca.components_)
np.save(RESULTS_ROOT / 'training' / EXPERIMENT_ID / 'weights' / f'{output_prefix}_pca_mean.npy', pca.mean_)